# Exploratory Actuarial Analysis

## General Insurance Capital & Risk Modelling

### Anthony Utulu Analytics

## Purpose

This notebook investigates the observed risk characteristics of the motor
insurance portfolio before statistical modelling.

The analysis is targeted at four areas:

1. Claim frequency
2. Claim severity
3. Tail risk
4. Risk segmentation

The objective is not to explore every available variable, but to identify
patterns that are relevant to underwriting risk, aggregate losses and the
subsequent capital modelling framework.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
project_dir = Path.cwd().parent
data_dir = project_dir / "data" / "processed"

# Load validated datasets
freq = pd.read_csv(data_dir / "freMTPL2freq.csv")
sev = pd.read_csv(data_dir / "freMTPL2sev.csv")

print(f"Frequency dataset: {freq.shape}")
print(f"Severity dataset: {sev.shape}")

Frequency dataset: (677991, 12)
Severity dataset: (26444, 2)


## 1. Portfolio Risk Baseline

Before examining individual risk characteristics, the overall portfolio
experience is established using exposure, claim frequency, claim severity
and observed loss cost.

In [2]:
total_exposure = freq["Exposure"].sum()
total_claims = freq["ClaimNb"].sum()
total_loss = sev["ClaimAmount"].sum()

claim_frequency = total_claims / total_exposure
average_severity = total_loss / total_claims
pure_premium = total_loss / total_exposure

print(f"Total exposure: {total_exposure:,.2f} policy-years")
print(f"Total claims: {total_claims:,}")
print(f"Total observed losses: €{total_loss:,.2f}")
print(f"Claim frequency: {claim_frequency:.4f} claims per policy-year")
print(f"Average claim severity: €{average_severity:,.2f}")
print(f"Observed pure premium: €{pure_premium:,.2f} per policy-year")

Total exposure: 358,482.84 policy-years
Total claims: 26,444
Total observed losses: €59,909,216.50
Claim frequency: 0.0738 claims per policy-year
Average claim severity: €2,265.51
Observed pure premium: €167.12 per policy-year


### Baseline Interpretation

The portfolio contains 358,482.84 policy-years of exposure and 26,444
reported claims, producing an observed claim frequency of approximately
0.0738 claims per policy-year.

Total observed claim cost is approximately €59.91 million, with an average
claim severity of €2,265.51. Combining the observed frequency and severity
produces a pure premium of approximately €167.12 per policy-year.

These portfolio averages provide the empirical baseline for subsequent
frequency, severity and aggregate-loss modelling. However, averages alone
do not describe the variability or extreme-loss behaviour that drives
capital risk. The next stage therefore investigates how claim frequency
varies across relevant portfolio characteristics.

## 2. Frequency Risk Analysis

### 2.1 Claim Frequency by Area

Claim frequency is evaluated relative to policy exposure rather than policy
count. This allows differences in the duration of risk exposure to be
properly reflected.

The first segmentation examines whether observed claim frequency varies
across the six geographical area categories.

In [3]:
area_frequency = (
    freq.groupby("Area", observed=True)
        .agg(
            Policies=("IDpol", "count"),
            Exposure=("Exposure", "sum"),
            Claims=("ClaimNb", "sum")
        )
)

area_frequency["ClaimFrequency"] = (
    area_frequency["Claims"] / area_frequency["Exposure"]
)

area_frequency["ExposureShare"] = (
    area_frequency["Exposure"] / total_exposure
)

area_frequency.round({
    "Exposure": 2,
    "ClaimFrequency": 4,
    "ExposureShare": 4
})

,Policies,Exposure,Claims,ClaimFrequency,ExposureShare
Area,,,,,
A,103952,61966.14,3364,0.0543,0.1729
B,75457,43010.75,2633,0.0612,0.1200
C,191874,104443.16,7093,0.0679,0.2913
D,151592,77116.92,6458,0.0837,0.2151
E,137163,63817.10,6122,0.0959,0.1780
F,17953,8128.75,774,0.0952,0.0227


### Interpretation

Observed claim frequency varies materially across geographical areas.

Frequency increases from 0.0543 claims per policy-year in Area A to 0.0959
in Area E. Area E therefore exhibits approximately 77% higher observed
claim frequency than Area A.

The higher-frequency areas are also commercially relevant. Areas D and E
represent approximately 21.5% and 17.8% of total portfolio exposure,
respectively. Area F has a similarly high observed frequency, but represents
only 2.3% of exposure and therefore has substantially less influence on
aggregate portfolio risk.

These results indicate geographical heterogeneity in claim frequency and
support the inclusion of Area as a candidate risk factor in subsequent
frequency modelling. At this exploratory stage, the results are descriptive
and should not be interpreted as evidence of causation.

### 2.2 Claim Frequency by Driver Age

Driver age may influence claim frequency because driving experience, behaviour and risk characteristics can vary across age groups.

For exploratory analysis, driver ages are grouped into interpretable bands. Claim frequency is calculated using exposure as the denominator so that differences in time at risk are appropriately reflected.

In [4]:
# Create driver age bands

age_bins = [17, 25, 35, 45, 55, 65, 75, 101]
age_labels = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65-74",
    "75+"
]

freq["DriverAgeBand"] = pd.cut(
    freq["DrivAge"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

# Calculate claim frequency by driver age band

age_frequency = (
    freq.groupby("DriverAgeBand", observed=True)
        .agg(
            Policies=("IDpol", "count"),
            Exposure=("Exposure", "sum"),
            Claims=("ClaimNb", "sum")
        )
)

age_frequency["ClaimFrequency"] = (
    age_frequency["Claims"] / age_frequency["Exposure"]
)

age_frequency["ExposureShare"] = (
    age_frequency["Exposure"] / total_exposure
)

age_frequency.round({
    "Exposure": 2,
    "ClaimFrequency": 4,
    "ExposureShare": 4
})

,Policies,Exposure,Claims,ClaimFrequency,ExposureShare
DriverAgeBand,,,,,
18-24,30196,12487.37,2011,0.1610,0.0348
25-34,140944,64597.36,4973,0.0770,0.1802
35-44,170573,87386.83,6192,0.0709,0.2438
45-54,164028,89848.00,6802,0.0757,0.2506
55-64,99091,56174.49,3602,0.0641,0.1567
65-74,50818,32205.45,1872,0.0581,0.0898
75+,22341,15783.33,992,0.0629,0.0440


### Interpretation

Claim frequency varies materially across driver age groups.

Drivers aged 18–24 exhibit the highest observed claim frequency at 0.1610 claims
per policy-year. This is substantially higher than all other age groups and is
more than twice the frequency observed for several mature driver segments.

Frequency falls sharply after age 24 and generally remains between approximately
0.058 and 0.077 across the remaining age groups. The lowest observed frequency
is 0.0581 for drivers aged 65–74, followed by a modest increase to 0.0629 for
drivers aged 75 and above.

The 18–24 segment represents only 3.5% of total portfolio exposure, limiting its
contribution to aggregate exposure despite its high individual frequency risk.
By contrast, drivers aged 35–44 and 45–54 account for approximately 24.4% and
25.1% of exposure respectively, making their risk experience particularly
important to overall portfolio performance.

The results indicate that driver age is a potentially important frequency risk
factor and should be considered in subsequent frequency modelling. These
observed differences are descriptive and may also reflect interactions with
other portfolio characteristics.

### 2.3 Claim Frequency by Bonus-Malus

The Bonus-Malus variable provides information about the policyholder's insurance
risk classification and may capture differences in prior claims experience.

For exploratory analysis, Bonus-Malus values are grouped into interpretable bands
to assess whether observed claim frequency changes across risk classifications.

In [5]:
# Create Bonus-Malus bands

bm_bins = [49, 60, 80, 100, 120, 150, float("inf")]
bm_labels = [
    "50-59",
    "60-79",
    "80-99",
    "100-119",
    "120-149",
    "150+"
]

freq["BonusMalusBand"] = pd.cut(
    freq["BonusMalus"],
    bins=bm_bins,
    labels=bm_labels,
    right=True
)

# Calculate claim frequency by Bonus-Malus band

bm_frequency = (
    freq.groupby("BonusMalusBand", observed=True)
        .agg(
            Policies=("IDpol", "count"),
            Exposure=("Exposure", "sum"),
            Claims=("ClaimNb", "sum")
        )
)

bm_frequency["ClaimFrequency"] = (
    bm_frequency["Claims"] / bm_frequency["Exposure"]
)

bm_frequency["ExposureShare"] = (
    bm_frequency["Exposure"] / total_exposure
)

bm_frequency.round({
    "Exposure": 2,
    "ClaimFrequency": 4,
    "ExposureShare": 4
})

,Policies,Exposure,Claims,ClaimFrequency,ExposureShare
BonusMalusBand,,,,,
50-59,478585,273205.72,15168,0.0555,0.7621
60-79,118752,53063.42,5914,0.1115,0.1480
80-99,72860,28642.38,4106,0.1434,0.0799
100-119,5554,2573.00,800,0.3109,0.0072
120-149,2031,903.19,402,0.4451,0.0025
150+,209,95.13,54,0.5677,0.0003


### Interpretation

Observed claim frequency increases strongly as the Bonus-Malus level rises.

The 50–59 band, which represents approximately 76.2% of total portfolio
exposure, has the lowest observed claim frequency at 0.0555 claims per
policy-year. Frequency increases to 0.1115 for the 60–79 band and 0.1434
for the 80–99 band.

A much steeper increase is observed at higher Bonus-Malus levels, reaching
0.3109 for 100–119, 0.4451 for 120–149 and 0.5677 for 150+.

The highest Bonus-Malus bands therefore exhibit substantially greater
observed frequency risk. However, these groups contain relatively little
exposure: the 100+ bands collectively represent approximately 1.0% of total
portfolio exposure, while the 150+ group represents only around 0.03%.
Their extreme observed frequencies should therefore be interpreted alongside
their limited exposure.

The clear progression across Bonus-Malus bands suggests that Bonus-Malus is
a strong candidate risk factor for subsequent frequency modelling. The
exploratory relationship is descriptive and will require multivariate
modelling to assess its effect after controlling for other portfolio
characteristics.

### 2.4 Claim Frequency by Vehicle Age

Vehicle age may influence claim frequency through differences in vehicle usage,
condition, technology and the characteristics of policyholders associated with
different vehicles.

Vehicle ages are grouped into interpretable bands to examine how claim frequency
and portfolio exposure vary across the vehicle-age distribution.

In [6]:
# Create vehicle age bands

vehicle_age_bins = [-1, 3, 6, 11, 16, 21, 31, float("inf")]
vehicle_age_labels = [
    "0-2",
    "3-5",
    "6-10",
    "11-15",
    "16-20",
    "21-30",
    "31+"
]

freq["VehicleAgeBand"] = pd.cut(
    freq["VehAge"],
    bins=vehicle_age_bins,
    labels=vehicle_age_labels,
    right=False
)

# Calculate claim frequency by vehicle age band

vehicle_age_frequency = (
    freq.groupby("VehicleAgeBand", observed=True)
        .agg(
            Policies=("IDpol", "count"),
            Exposure=("Exposure", "sum"),
            Claims=("ClaimNb", "sum")
        )
)

vehicle_age_frequency["ClaimFrequency"] = (
    vehicle_age_frequency["Claims"] /
    vehicle_age_frequency["Exposure"]
)

vehicle_age_frequency["ExposureShare"] = (
    vehicle_age_frequency["Exposure"] /
    total_exposure
)

vehicle_age_frequency.round({
    "Exposure": 2,
    "ClaimFrequency": 4,
    "ExposureShare": 4
})

,Policies,Exposure,Claims,ClaimFrequency,ExposureShare
VehicleAgeBand,,,,,
0-2,188140,79171.14,5747,0.0726,0.2209
3-5,132488,72038.12,5344,0.0742,0.2010
6-10,171587,99003.24,8009,0.0809,0.2762
11-15,134059,77988.00,5680,0.0728,0.2176
16-20,43396,25135.23,1457,0.0580,0.0701
21-30,7205,4422.59,186,0.0421,0.0123
31+,1116,724.52,21,0.0290,0.0020


### Interpretation

Observed claim frequency varies across vehicle-age groups, although the
relationship is not strictly monotonic.

Vehicles aged 6–10 years exhibit the highest observed claim frequency at
0.0809 claims per policy-year. This segment is particularly relevant because
it also represents the largest share of portfolio exposure at approximately
27.6%.

Vehicles aged 0–2, 3–5 and 11–15 years show relatively similar frequencies,
ranging from approximately 0.073 to 0.074 claims per policy-year.

Observed frequency declines among older vehicles, falling to 0.0580 for
vehicles aged 16–20 years and 0.0421 for vehicles aged 21–30 years. The
31+ group records the lowest frequency at 0.0290, although this segment
represents only 0.2% of total exposure and should therefore be interpreted
with caution.

Overall, vehicle age shows evidence of frequency differentiation and should
be retained as a candidate risk factor for subsequent modelling. The
non-linear pattern also suggests that its relationship with claim frequency
should not automatically be assumed to be linear.

### 2.5 Claim Frequency by Vehicle Power

Vehicle power may be associated with differences in driving behaviour,
vehicle characteristics and accident risk.

Observed claim frequency is examined across the available vehicle-power
categories together with their respective exposure shares.

In [7]:
# Calculate claim frequency by vehicle power

power_frequency = (
    freq.groupby("VehPower")
        .agg(
            Policies=("IDpol", "count"),
            Exposure=("Exposure", "sum"),
            Claims=("ClaimNb", "sum")
        )
)

power_frequency["ClaimFrequency"] = (
    power_frequency["Claims"] /
    power_frequency["Exposure"]
)

power_frequency["ExposureShare"] = (
    power_frequency["Exposure"] /
    total_exposure
)

power_frequency.round({
    "Exposure": 2,
    "ClaimFrequency": 4,
    "ExposureShare": 4
})

,Policies,Exposure,Claims,ClaimFrequency,ExposureShare
VehPower,,,,,
4,115344,60069.71,4003,0.0666,0.1676
5,124815,68168.82,5115,0.0750,0.1902
6,148972,82520.44,6305,0.0764,0.2302
7,145398,77948.44,5593,0.0718,0.2174
8,46954,22682.95,1645,0.0725,0.0633
9,30084,15348.11,1220,0.0795,0.0428
10,31353,15374.42,1217,0.0792,0.0429
11,18352,8497.78,714,0.0840,0.0237
12,8214,3792.04,296,0.0781,0.0106


### Interpretation

Claim frequency shows some variation across vehicle-power categories, although
the relationship is less pronounced than the patterns observed for Driver Age
and Bonus-Malus.

The majority of portfolio exposure is concentrated in vehicle powers 4–8,
which collectively represent approximately 87% of total exposure. Within these
high-exposure categories, observed claim frequency ranges from 0.0666 to 0.0764
claims per policy-year.

Higher vehicle-power categories generally show somewhat higher observed
frequencies. Frequencies reach 0.0840 for power 11, 0.0849 for power 13 and
0.0854 for power 14. However, these categories represent relatively small
shares of total exposure, so their results should be interpreted with greater
caution.

Overall, vehicle power demonstrates some frequency differentiation and should
remain a candidate explanatory variable for subsequent modelling. The
exploratory evidence is weaker than that observed for Driver Age and
Bonus-Malus, and the relationship does not appear strictly monotonic.

### 2.6 Claim Frequency by Fuel Type and Region

The final frequency-risk assessment considers vehicle fuel type and geographical
region.

Fuel type provides a broad vehicle characteristic, while Region allows the
analysis to assess whether geographical variation exists at a more detailed
level than the Area classification examined earlier.

These variables are evaluated using exposure-adjusted claim frequency together
with exposure share to distinguish risk differences from portfolio concentration.

In [8]:
# Claim frequency by vehicle fuel type

gas_frequency = (
    freq.groupby("VehGas")
        .agg(
            Policies=("IDpol", "count"),
            Exposure=("Exposure", "sum"),
            Claims=("ClaimNb", "sum")
        )
)

gas_frequency["ClaimFrequency"] = (
    gas_frequency["Claims"] / gas_frequency["Exposure"]
)

gas_frequency["ExposureShare"] = (
    gas_frequency["Exposure"] / total_exposure
)

print("CLAIM FREQUENCY BY VEHICLE FUEL TYPE")
display(
    gas_frequency.round({
        "Exposure": 2,
        "ClaimFrequency": 4,
        "ExposureShare": 4
    })
)

CLAIM FREQUENCY BY VEHICLE FUEL TYPE


,Policies,Exposure,Claims,ClaimFrequency,ExposureShare
VehGas,,,,,
Diesel,332128,170653.97,13450,0.0788,0.476
Regular,345863,187828.86,12994,0.0692,0.524


### 2.7 Claim Frequency by Region

Regional claim frequency is examined to identify geographical variation in
observed insurance risk.

Because the portfolio contains 22 regions, results are ranked by claim
frequency. Exposure share is retained alongside frequency so that unusually
high or low results can be interpreted in the context of portfolio credibility.

In [9]:
# Calculate claim frequency by region

region_frequency = (
    freq.groupby("Region")
        .agg(
            Policies=("IDpol", "count"),
            Exposure=("Exposure", "sum"),
            Claims=("ClaimNb", "sum")
        )
)

region_frequency["ClaimFrequency"] = (
    region_frequency["Claims"] / region_frequency["Exposure"]
)

region_frequency["ExposureShare"] = (
    region_frequency["Exposure"] / total_exposure
)

region_frequency = region_frequency.sort_values(
    "ClaimFrequency",
    ascending=False
)

region_frequency.round({
    "Exposure": 2,
    "ClaimFrequency": 4,
    "ExposureShare": 4
})

,Policies,Exposure,Claims,ClaimFrequency,ExposureShare
Region,,,,,
Rhone-Alpes,84748,45343.55,4233,0.0934,0.1265
Picardie,7994,3574.41,314,0.0878,0.0100
Ile-de-France,69789,30207.16,2591,0.0858,0.0843
Provence-Alpes-Cotes-D'Azur,79315,35790.32,2986,0.0834,0.0998
Limousin,4567,2395.99,197,0.0822,0.0067
Nord-Pas-de-Calais,27284,11496.97,944,0.0821,0.0321
Alsace,2200,1209.12,92,0.0761,0.0034
Corse,4516,1765.94,132,0.0747,0.0049
Aquitaine,31329,14322.17,1055,0.0737,0.0400


#### Interpretation

Claim frequency varies materially across regions, indicating that geography may
be an important component of portfolio risk.

Rhone-Alpes records the highest observed frequency among regions with substantial
portfolio exposure, at approximately 0.0934 claims per policy-year and 12.65% of
total exposure. In contrast, Centre represents the largest geographical
concentration, accounting for approximately 28.65% of exposure, while recording
a lower claim frequency of 0.0630.

Some regions with relatively high or low observed frequencies have limited
exposure and should therefore be interpreted cautiously. These results provide
exploratory evidence of geographical risk differentiation but do not establish
causality or adjusted risk effects.

Regional effects will be assessed alongside other risk characteristics during
the frequency-modelling stage.

## 3. Severity Risk Analysis

Claim severity represents the financial cost of claims once they occur. While
frequency determines how often losses arise, severity determines the magnitude
of those losses and is therefore particularly important when assessing extreme
insurance risk and capital requirements.

This section examines:

- the overall distribution of individual claim amounts;
- central and tail characteristics of claim severity;
- the concentration of losses in large claims;
- severity differences across selected portfolio risk segments; and
- implications for subsequent severity modelling and aggregate loss simulation.

Particular attention is given to the upper tail of the claim distribution,
because infrequent but very large claims can materially influence an insurer's
capital requirement.

In [10]:
# Overall claim severity profile

severity_summary = sev["ClaimAmount"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
)

print("CLAIM SEVERITY SUMMARY")
print(severity_summary)

print("\nAdditional severity metrics:")
print(f"Total claim cost: €{sev['ClaimAmount'].sum():,.2f}")
print(f"Average severity: €{sev['ClaimAmount'].mean():,.2f}")
print(f"Median severity: €{sev['ClaimAmount'].median():,.2f}")
print(f"Maximum claim: €{sev['ClaimAmount'].max():,.2f}")

CLAIM SEVERITY SUMMARY
count    2.644400e+04
mean     2.265513e+03
std      2.937103e+04
min      1.000000e+00
50%      1.172000e+03
75%      1.212385e+03
90%      2.767604e+03
95%      4.765093e+03
99%      1.645122e+04
99.5%    3.437696e+04
99.9%    1.522232e+05
max      4.075401e+06
Name: ClaimAmount, dtype: float64

Additional severity metrics:
Total claim cost: €59,909,216.50
Average severity: €2,265.51
Median severity: €1,172.00
Maximum claim: €4,075,400.56


### 3.1 Large-Loss Concentration

The severity distribution indicates substantial right-tail risk. The following
analysis measures how much of the portfolio's total observed claim cost is
concentrated within the largest claims.

This is important from a capital perspective because a relatively small number
of severe claims may account for a disproportionate share of aggregate losses.

In [11]:
# Measure concentration of total losses in the largest claims

claims_sorted = sev["ClaimAmount"].sort_values(ascending=False).reset_index(drop=True)

total_loss = claims_sorted.sum()
n_claims = len(claims_sorted)

concentration_results = []

for proportion in [0.001, 0.005, 0.01, 0.05, 0.10]:
    n_top = max(1, int(np.ceil(n_claims * proportion)))
    top_loss = claims_sorted.iloc[:n_top].sum()

    concentration_results.append({
        "TopClaims": f"{proportion * 100:g}%",
        "NumberOfClaims": n_top,
        "LossAmount": top_loss,
        "ShareOfTotalLoss": top_loss / total_loss
    })

loss_concentration = pd.DataFrame(concentration_results)

loss_concentration["LossAmount"] = (
    loss_concentration["LossAmount"].round(2)
)

loss_concentration["ShareOfTotalLoss"] = (
    loss_concentration["ShareOfTotalLoss"].round(4)
)

loss_concentration

,TopClaims,NumberOfClaims,LossAmount,ShareOfTotalLoss
0,0.1%,27,12959412.53,0.2163
1,0.5%,133,19743176.05,0.3296
2,1%,265,22773883.74,0.3801
3,5%,1323,31208589.23,0.5209
4,10%,2645,35901267.75,0.5993


#### Interpretation

Losses are highly concentrated in the upper tail of the claim severity
distribution. The largest 0.1% of claims, representing only 27 individual
claims, account for approximately 21.63% of total observed losses.

The largest 1% of claims account for approximately 38.01% of total loss cost,
while the largest 5% account for more than half (52.09%) of all observed losses.

This demonstrates that aggregate portfolio losses are materially influenced by
a relatively small number of severe claims. Consequently, relying only on
average claim severity would understate the importance of large-loss risk.

The result is particularly relevant to capital modelling because the insurer's
financial position may be sensitive to the occurrence and magnitude of extreme
claims. The upper tail will therefore require explicit consideration during
severity distribution fitting, aggregate loss simulation and subsequent
capital-risk assessment.

### 3.2 Claim Severity by Portfolio Risk Segment

Individual claim amounts are linked to the corresponding policy characteristics
using the policy identifier (`IDpol`). This allows observed claim severity to be
examined across relevant portfolio segments.

The objective is to determine whether the magnitude of claims varies materially
across risk characteristics and to identify segmentation that may be relevant
for subsequent severity modelling and capital simulation.

In [12]:
# Link individual claims to policy risk characteristics

severity_policy = sev.merge(
    freq,
    on="IDpol",
    how="left",
    validate="many_to_one"
)

print("SEVERITY-POLICY JOIN VALIDATION")
print(f"Original severity records: {len(sev):,}")
print(f"Records after join:       {len(severity_policy):,}")

unmatched = severity_policy["Exposure"].isna().sum()
print(f"Unmatched claim records:  {unmatched:,}")

print("\nColumns available:")
print(severity_policy.columns.tolist())

SEVERITY-POLICY JOIN VALIDATION
Original severity records: 26,444
Records after join:       26,444
Unmatched claim records:  0

Columns available:
['IDpol', 'ClaimAmount', 'ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Area', 'Density', 'Region', 'DriverAgeBand', 'BonusMalusBand', 'VehicleAgeBand']


#### 3.3 Claim Severity by Area

Claim severity is examined across geographical area classifications to determine
whether areas associated with different claim frequencies also exhibit
differences in the financial magnitude of individual claims.

Both mean and median severity are considered because the claim distribution is
strongly right-skewed and the mean may be materially influenced by large losses.

In [13]:
# Claim severity by area

area_severity = (
    severity_policy
    .groupby("Area", observed=True)
    .agg(
        Claims=("ClaimAmount", "size"),
        TotalLoss=("ClaimAmount", "sum"),
        MeanSeverity=("ClaimAmount", "mean"),
        MedianSeverity=("ClaimAmount", "median"),
        MaxClaim=("ClaimAmount", "max")
    )
)

area_severity["LossShare"] = (
    area_severity["TotalLoss"] / total_loss
)

area_severity.round({
    "TotalLoss": 2,
    "MeanSeverity": 2,
    "MedianSeverity": 2,
    "MaxClaim": 2,
    "LossShare": 4
})

,Claims,TotalLoss,MeanSeverity,MedianSeverity,MaxClaim,LossShare
Area,,,,,,
A,3364,7739630.67,2300.72,1172.0,369131.88,0.1292
B,2633,8873979.52,3370.29,1172.0,4075400.56,0.1481
C,7093,14612072.43,2060.07,1172.0,1403057.40,0.2439
D,6458,14486501.16,2243.19,1172.0,1301172.60,0.2418
E,6122,13017426.30,2126.34,1172.0,774411.50,0.2173
F,774,1179606.42,1524.04,1172.0,114759.95,0.0197


#### Interpretation

Observed mean claim severity varies across geographical areas, but the results
are strongly influenced by large individual losses.

Area B records the highest mean severity at approximately €3,370, compared with
the portfolio average of approximately €2,266. However, Area B also contains
the portfolio's largest individual claim of approximately €4.08 million.

Median severity is €1,172 across all six area classifications, indicating that
the typical claim size is remarkably similar across areas. The much greater
variation in mean severity therefore appears to be driven primarily by the
upper tail rather than broad differences in ordinary claim sizes.

Areas C and D account for the largest shares of total observed losses, at
approximately 24.39% and 24.18%, respectively, but they also contain substantial
claim volumes.

These results reinforce the importance of modelling the severity distribution
and its upper tail rather than relying solely on segment-level mean severity.

In [14]:
# Create consistent driver age bands for severity analysis

severity_policy["DriverAgeBand"] = pd.cut(
    severity_policy["DrivAge"],
    bins=[17, 24, 34, 44, 54, 64, 74, np.inf],
    labels=["18-24", "25-34", "35-44", "45-54",
            "55-64", "65-74", "75+"]
)

driver_age_severity = (
    severity_policy
    .groupby("DriverAgeBand", observed=True)
    .agg(
        Claims=("ClaimAmount", "size"),
        TotalLoss=("ClaimAmount", "sum"),
        MeanSeverity=("ClaimAmount", "mean"),
        MedianSeverity=("ClaimAmount", "median"),
        MaxClaim=("ClaimAmount", "max")
    )
)

driver_age_severity["LossShare"] = (
    driver_age_severity["TotalLoss"] / total_loss
)

driver_age_severity.round({
    "TotalLoss": 2,
    "MeanSeverity": 2,
    "MedianSeverity": 2,
    "MaxClaim": 2,
    "LossShare": 4
})

,Claims,TotalLoss,MeanSeverity,MedianSeverity,MaxClaim,LossShare
DriverAgeBand,,,,,,
18-24,2011,11741344.21,5838.56,1172.00,4075400.56,0.1960
25-34,4973,9957419.91,2002.30,1128.12,276643.00,0.1662
35-44,6192,12284481.34,1983.93,1167.46,774411.50,0.2051
45-54,6802,12569823.09,1847.96,1172.00,301635.49,0.2098
55-64,3602,6697343.57,1859.34,1172.00,390742.27,0.1118
65-74,1872,4123771.44,2202.87,1172.00,211204.00,0.0688
75+,992,2535032.94,2555.48,1204.00,287423.00,0.0423


#### Interpretation

Driver age reveals an important interaction between claim frequency and
severity risk.

Drivers aged 18–24 have an observed mean severity of approximately €5,839,
substantially above all other age groups. This group also recorded the highest
claim frequency in the portfolio at approximately 0.161 claims per policy-year.

However, the severity result is strongly influenced by the portfolio's largest
individual claim of approximately €4.08 million, which falls within the 18–24
age band. Median severity for this group is €1,172, broadly consistent with the
other age groups. The elevated mean should therefore be interpreted as evidence
of observed tail exposure rather than a uniformly higher typical claim size.

The 18–24 segment is nevertheless important from a capital-risk perspective
because the observed data combines high claim frequency with exposure to a
very large loss. This distinction between expected claim behaviour and
low-frequency, high-severity outcomes will be considered explicitly during
frequency, severity and aggregate-loss modelling.

In [15]:
# Create consistent Bonus-Malus bands for severity analysis

severity_policy["BonusMalusBand"] = pd.cut(
    severity_policy["BonusMalus"],
    bins=[49, 59, 79, 99, 119, 149, np.inf],
    labels=["50-59", "60-79", "80-99",
            "100-119", "120-149", "150+"]
)

bm_severity = (
    severity_policy
    .groupby("BonusMalusBand", observed=True)
    .agg(
        Claims=("ClaimAmount", "size"),
        TotalLoss=("ClaimAmount", "sum"),
        MeanSeverity=("ClaimAmount", "mean"),
        MedianSeverity=("ClaimAmount", "median"),
        MaxClaim=("ClaimAmount", "max")
    )
)

bm_severity["LossShare"] = bm_severity["TotalLoss"] / total_loss

bm_severity.round({
    "TotalLoss": 2,
    "MeanSeverity": 2,
    "MedianSeverity": 2,
    "MaxClaim": 2,
    "LossShare": 4
})

,Claims,TotalLoss,MeanSeverity,MedianSeverity,MaxClaim,LossShare
BonusMalusBand,,,,,,
50-59,14626,28329318.48,1936.91,1172.0,390742.27,0.4729
60-79,5750,11716035.01,2037.57,1172.0,774411.50,0.1956
80-99,3492,7116884.87,2038.05,1172.0,307096.42,0.1188
100-119,2119,11747623.90,5543.95,1172.0,4075400.56,0.1961
120-149,403,866677.83,2150.57,1172.0,205132.00,0.0145
150+,54,132676.41,2456.97,1172.0,39071.07,0.0022


#### Interpretation

Bonus-Malus shows a much stronger relationship with claim frequency than with
typical claim severity.

Earlier frequency analysis showed a pronounced increase in claim frequency as
Bonus-Malus increased. In contrast, median claim severity is approximately
€1,172 across all Bonus-Malus bands, providing little evidence of a systematic
increase in the typical size of individual claims.

The 100–119 band records the highest observed mean severity at approximately
€5,544. However, this segment contains the portfolio's €4.08 million maximum
claim, which materially increases its mean. The difference between the mean
and median therefore highlights the influence of extreme losses.

From a capital-risk perspective, Bonus-Malus appears particularly useful for
distinguishing claim occurrence risk, while severity risk remains dominated
by the heavy upper tail of the claim-size distribution. Frequency and severity
should therefore be modelled separately rather than assuming that segments
with higher claim frequency necessarily generate proportionately larger
individual claims.

In [16]:
# Create consistent vehicle age bands for severity analysis

severity_policy["VehicleAgeBand"] = pd.cut(
    severity_policy["VehAge"],
    bins=[-1, 2, 5, 10, 15, 20, 30, np.inf],
    labels=["0-2", "3-5", "6-10", "11-15",
            "16-20", "21-30", "31+"]
)

vehicle_age_severity = (
    severity_policy
    .groupby("VehicleAgeBand", observed=True)
    .agg(
        Claims=("ClaimAmount", "size"),
        TotalLoss=("ClaimAmount", "sum"),
        MeanSeverity=("ClaimAmount", "mean"),
        MedianSeverity=("ClaimAmount", "median"),
        MaxClaim=("ClaimAmount", "max")
    )
)

vehicle_age_severity["LossShare"] = (
    vehicle_age_severity["TotalLoss"] / total_loss
)

vehicle_age_severity.round({
    "TotalLoss": 2,
    "MeanSeverity": 2,
    "MedianSeverity": 2,
    "MaxClaim": 2,
    "LossShare": 4
})

,Claims,TotalLoss,MeanSeverity,MedianSeverity,MaxClaim,LossShare
VehicleAgeBand,,,,,,
0-2,5747,12299662.34,2140.19,1172.00,390742.27,0.2053
3-5,5344,10552899.23,1974.72,1172.00,301635.49,0.1761
6-10,8009,15805071.87,1973.41,1159.53,774411.50,0.2638
11-15,5680,17808346.08,3135.27,1128.12,4075400.56,0.2973
16-20,1457,3010392.67,2066.16,1172.00,287423.00,0.0502
21-30,186,407597.28,2191.38,1172.00,182500.00,0.0068
31+,21,25247.03,1202.24,1172.00,4172.00,0.0004


#### Interpretation

Claim severity does not display a clear monotonic relationship with vehicle age.

Median claim severity remains relatively stable across the main vehicle-age
segments, generally close to €1,100–€1,200. Mean severity, however, varies more
substantially because individual large losses influence the segment averages.

The 11–15 year vehicle segment records the highest mean severity at approximately
€3,135 and contributes 29.73% of total observed losses. This result is materially
influenced by the portfolio's €4.08 million maximum claim, which occurs within
this segment. It should therefore not be interpreted as evidence that vehicles
aged 11–15 years systematically generate claims of this magnitude.

The 6–10 and 11–15 year groups together account for approximately 56.1% of total
observed losses, reflecting both their substantial portfolio presence and their
claims experience.

From a capital-risk perspective, the results again demonstrate that extreme
individual losses can materially affect aggregate loss experience. Vehicle age
may help explain portfolio composition and claim occurrence, but the upper tail
of the severity distribution remains a more important consideration for
unexpected-loss modelling.

In [17]:
# Claim severity by vehicle power

power_severity = (
    severity_policy
    .groupby("VehPower")
    .agg(
        Claims=("ClaimAmount", "size"),
        TotalLoss=("ClaimAmount", "sum"),
        MeanSeverity=("ClaimAmount", "mean"),
        MedianSeverity=("ClaimAmount", "median"),
        MaxClaim=("ClaimAmount", "max")
    )
)

power_severity["LossShare"] = (
    power_severity["TotalLoss"] / total_loss
)

power_severity.round({
    "TotalLoss": 2,
    "MeanSeverity": 2,
    "MedianSeverity": 2,
    "MaxClaim": 2,
    "LossShare": 4
})

,Claims,TotalLoss,MeanSeverity,MedianSeverity,MaxClaim,LossShare
VehPower,,,,,,
4,4003,9410083.43,2350.76,1172.00,1301172.60,0.1571
5,5115,8837747.14,1727.81,1172.00,211204.00,0.1475
6,6305,14385946.83,2281.67,1161.32,1403057.40,0.2401
7,5593,12426397.16,2221.78,1172.00,774411.50,0.2074
8,1645,2954665.51,1796.15,1172.00,115232.88,0.0493
9,1220,6265052.65,5135.29,1172.00,4075400.56,0.1046
10,1217,2853312.33,2344.55,1172.00,205432.05,0.0476
11,714,1312132.16,1837.72,1204.00,116318.90,0.0219
12,296,707682.28,2390.82,1204.00,109743.72,0.0118


#### Interpretation

Claim severity does not show a clear or consistent relationship with vehicle
power.

Median severity remains broadly stable across vehicle-power groups, generally
close to €1,100–€1,200. Mean severity varies more substantially because several
segments contain individual large losses.

Vehicle power 9 records the highest mean severity at approximately €5,135, but
this segment contains the portfolio's €4.08 million maximum claim. Its elevated
mean is therefore largely attributable to tail-loss experience rather than a
general increase in the typical claim size.

Vehicle powers 6 and 7 account for approximately 24.0% and 20.7% of total
observed losses respectively. However, these groups also contain substantial
numbers of claims, so their contribution reflects portfolio volume as well as
severity.

Overall, vehicle power appears less informative for explaining typical claim
severity than the heavy-tailed nature of the claim-size distribution. This
supports modelling claim severity probabilistically rather than relying on
simple segment averages for capital-risk assessment.

## 4. Exploratory Risk Synthesis

The exploratory analysis identifies two materially different components of
insurance risk: claim frequency risk and claim severity risk.

### Frequency Risk

Observed portfolio claim frequency is approximately 0.0738 claims per
policy-year, but the risk is not homogeneous across the portfolio.

The strongest segmentation signal is Bonus-Malus. Claim frequency rises from
approximately 0.0555 in the 50–59 band to 0.5677 in the 150+ band. Driver age
also shows meaningful differentiation, with drivers aged 18–24 recording a
frequency of approximately 0.1610 compared with materially lower frequencies
for most older groups.

Geographical area, region, vehicle age, fuel type and vehicle power also show
differences in observed claim frequency. These patterns indicate that a single
portfolio-wide claim frequency assumption would fail to capture important
heterogeneity in claim occurrence risk.

### Severity Risk

Claim severity exhibits a substantially different risk structure. The median
claim is approximately €1,172, while mean severity is approximately €2,266.
The difference is caused by a strongly right-skewed claim-size distribution
containing a small number of very large losses.

The largest observed claim is approximately €4.08 million. The top 0.1% of
claims account for approximately 21.6% of total observed losses, while the top
1% account for approximately 38.0%.

Segment-level severity analysis shows that median severity remains relatively
stable across several portfolio characteristics. Large differences in mean
severity are frequently associated with individual extreme claims rather than
a consistent underlying segment effect.

### Capital-Risk Implication

The portfolio therefore contains two distinct sources of uncertainty.

Claim frequency risk is associated with systematic differences in the rate at
which claims occur across policyholder and vehicle characteristics.

Claim severity risk is dominated more strongly by the heavy upper tail of the
claim-size distribution and the possibility of infrequent but financially
material losses.

For capital modelling, frequency and severity should therefore be modelled
separately before being combined into an aggregate annual loss distribution.
The resulting distribution can then be used to evaluate expected losses,
unexpected losses and tail-risk measures such as Value at Risk and Expected
Shortfall.